In [1112]:
from models import *
from utilities import *

In [1113]:
import pickle
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import cm
import seaborn as sns
import copy
import math
import warnings
from scipy.stats import mannwhitneyu
warnings.filterwarnings('ignore')

In [1114]:
def remove_rows_within_n_days(data_copy: pd.DataFrame,
                             blood_trans: pd.DataFrame,
                             n: int) -> pd.DataFrame:
    # Make sure dates are datetime
    dc = data_copy.copy()
    dc['date'] = pd.to_datetime(dc['date'])
    bt = blood_trans.copy()
    bt['BloodTran_Start_DT'] = pd.to_datetime(bt['BloodTran_Start_DT'])

    # Keep original row index to identify rows later
    dc = dc.reset_index().rename(columns={'index': 'orig_idx'})

    # Merge to get all Start_DT for each SCCRIP_ID
    merged = dc.merge(bt[['id', 'BloodTran_Start_DT']], on='id', how='left')

    # Condition: Start_DT < date <= Start_DT + n days
    mask = (
        merged['BloodTran_Start_DT'].notna()
        & (merged['date'] > merged['BloodTran_Start_DT'])
        & (merged['date'] <= merged['BloodTran_Start_DT'] + pd.Timedelta(days=n))
    )

    # Find original rows that match the condition for ANY Start_DT
    bad_idx = merged.loc[mask, 'orig_idx'].unique()

    # Drop those rows from the original data_copy
    result = dc[~dc['orig_idx'].isin(bad_idx)].drop(columns='orig_idx')

    return result

def add_ctxn_flag(data_copy: pd.DataFrame,
                  ctxn: pd.DataFrame) -> pd.DataFrame:
    """
    Add a new column 'ctxn' to data_copy:
      ctxn = 1 if SCCRIP_ID in ctxn and date > CTXN_Start_DT1 (for that ID)
      ctxn = 0 otherwise.

    Assumes:
        - data_copy has ['SCCRIP_ID', 'date']
        - ctxn has ['SCCRIP_ID', 'CTXN_Start_DT1']
    """
    dc = data_copy.copy()
    cx = ctxn.copy()

    # Ensure datetime columns
    dc['date'] = pd.to_datetime(dc['date'])
    cx['CTXN_Start_DT1'] = pd.to_datetime(cx['CTXN_Start_DT1'])

    # Merge by SCCRIP_ID to attach CTXN_Start_DT1 to each data_copy row (if ID exists)
    merged = dc.merge(cx[['id', 'CTXN_Start_DT1']],
                      on='id', how='left')

    # Vectorized condition
    merged['ctxn'] = (
        (merged['CTXN_Start_DT1'].notna())
        & (merged['date'] > merged['CTXN_Start_DT1'])
    ).astype(int)

    # Drop the CTXN_Start_DT1 column if you don’t need it in the result
    merged = merged.drop(columns=['CTXN_Start_DT1'])

    return merged

In [1115]:
with open('../results/data/SCCRIP_ID_dict.pkl', 'rb') as f:
    mapping_dict = pickle.load(f)

In [1116]:
ctxn = read_sas('../data/Apply_MI_Zheng_2023_01/ctxn.sas7bdat')
ctxn['id'] = ctxn['SCCRIP_ID'].map(mapping_dict)
ctxn = ctxn[~ctxn['id'].isnull()]
ctxn = ctxn[['id', 'CTXN_Start_DT1']]
blood_trans = read_sas('../data/Apply_MI_Zheng_2023_01/bloodtrans_final.sas7bdat')
blood_trans['id'] = blood_trans['SCCRIP_ID'].map(mapping_dict)
blood_trans = blood_trans[~blood_trans['id'].isnull()]
blood_trans = blood_trans[['id', 'BloodTran_Start_DT']]

In [1117]:
dfs = pd.read_pickle('../results/data/all_treat_control_prediction_loop.pkl')

In [1118]:
df_hu = dfs[(dfs['hus'] > 0) & (dfs['hus'] < 100)]

In [1119]:
hu = read_sas('../data/Apply_MI_Zheng_2023_01/hu.sas7bdat')
hu['id'] = hu['SCCRIP_ID'].map(mapping_dict)
hu = hu[['id', 'HU_Start_DT1']]

In [1120]:
data_filter = remove_rows_within_n_days(df_hu, blood_trans, n = 30)
data_flag = add_ctxn_flag(data_filter, ctxn)

HU_all_c = data_flag[data_flag['ctxn']==0]
HU_all_c.shape

(14741, 8)

In [1121]:
######################################################################

In [1122]:
HU_all_c['err_t'] = HU_all_c['pred_t'] - HU_all_c['pred_c']
HU_all_c['err_c'] = HU_all_c['test'] - HU_all_c['pred_c']

In [1123]:
######################

In [1124]:
df_hus = HU_all_c[['id', 'age', 'hus','test', 'err_c','err_t', 'date']]

In [1125]:
df_for_gee = copy.deepcopy(df_hus)

In [1126]:
hos = read_sas('../data/Apply_MI_Zheng_2023_01/edhosp.sas7bdat')
hos['VisitDate'] = pd.to_datetime(hos['VisitDate'])
hos['ID'] = hos['SCCRIP_ID'].map(mapping_dict)
hos = hos[hos['Fin_VisitType'] == b'Inpatient']
hos = hos.fillna(0)

In [1127]:
cols = list(hos.columns)
i, j = cols.index('Viral_Infection'), cols.index('ACS')
cols[i], cols[j] = cols[j], cols[i]
hos = hos[cols]

In [1128]:
index1 = cols.index('Viral_Infection')
index2 = cols.index('Others')+1

In [1129]:
df_keep = hos.iloc[:, :index1]
new_col = (hos.iloc[:, index1:index2] == 1).any(axis=1).astype(int)
hos_new = df_keep.copy()
hos_new['Others'] = new_col
hos_new['ID'] = hos['ID']

In [1130]:
hos_new = hos_new[['ID', 'VisitDate', 'DischargeDate', 'Stay_days', 'VOC', 'ACS', 'FEVER', 'Others']]

In [1131]:
df_hos = copy.deepcopy(hos_new)

In [1132]:
d_windows, n_windows, n_per_year = 90,24,4

In [1133]:
def get_final_rows(t_df, t_hos, n,ids,d_windows=d_windows):
    df = pd.DataFrame(columns = ['n', 'id','age', 'date', 'hbf', 'err', 'err_t', 'hos_days', 'ACS', 'VOC', 'FEVER', 'Others'])
    
    s = t_df['date'].min()    
    for i in range(n): 
        s0 = s
        s = s + pd.Timedelta(days=d_windows)
        t = t_df[(t_df['date'] >= s0) & (t_df['date'] <= s)]
        h = t_hos[(t_hos['VisitDate'] >= s0) & (t_hos['VisitDate'] <= s)]
        if len(t) > 0:
            hbf = t['test'].mean()
            err = t['err_c'].mean()
            err_t = t['err_t'].mean()

            date = s0
            age = t.iloc[0]['age']
            hos_d = h['Stay_days'].sum()
            acs = h["ACS"].sum() 
            voc = h['VOC'].sum() 
            fever = h['FEVER'].sum()
            other = h['Others'].sum()
            df.loc[len(df)] = copy.deepcopy([i, ids, age, date, hbf, err, err_t, hos_d, acs, voc, fever, other])
            
    return df


In [1134]:
len(df_for_gee.id.unique())

575

In [1135]:
df_for_gee['age'] = df_for_gee['age'].astype('float')

In [1136]:
df_for_gee.age.max(), df_for_gee.age.min()

(61.27310061601643, 0.7227926078028748)

In [1137]:
id_list = set(df_hos.ID).intersection(set(df_for_gee.id))
                        
df_list = []
for i in id_list:
    t_hos = df_hos[df_hos['ID'] == i]
    t_df = df_for_gee[df_for_gee['id'] == i]
    t_df = t_df.sort_values(by='date')
    t_hos = t_hos.sort_values(by='VisitDate')

               
    n_win = min((t_df['date'].max() - t_df['date'].min()).days//d_windows, n_windows)
    n_win = (n_win//n_per_year)*n_per_year

    df_ = get_final_rows(t_df=t_df, t_hos=t_hos, n=n_win,ids=i,d_windows=d_windows)
    df_list.append(copy.deepcopy(df_))

In [1138]:
df_all = pd.concat(df_list)

In [1139]:
df_all = df_all.fillna(0)

In [1140]:
df_all = df_all.merge(hu, how='left')

In [1141]:
############ GEE

In [1142]:
import statsmodels.api as sm
import statsmodels.formula.api as smf

In [1143]:
df = copy.deepcopy(df_all)
df["date"] = pd.to_datetime(df["date"])
df["HU_Start_DT1"] = pd.to_datetime(df["HU_Start_DT1"])
df = df.sort_values(["id", "date"]).copy()

df["interval_years"] = (df["date"]-df['HU_Start_DT1']).dt.days/365
df = df.fillna(0)
df["voc_acs"] = np.where((df["VOC"] > 0) | (df["ACS"] > 0), df['hos_days'], 0)
df["voc"] = np.where(df["VOC"] > 0, df['hos_days'], 0)
df["acs"] = np.where(df["ACS"] > 0, df['hos_days'], 0)
df["fever"] = np.where(df["FEVER"] > 0, df['hos_days'], 0)

# HbF group at visit
df["low_hbf"] = (df["hbf"] <= 20).astype(int)
df["low_adj_hbf"] = (df["err"] <= 9).astype(int)
df["low_sim_hbf"] = (df["err_t"] <= 12).astype(int)

In [1144]:
def gee_pro(df, X_col, Y_col):
    df2 = df.copy()
    df2 = df2.dropna()

    if (df2["interval_years"] <= 0).any():
        raise ValueError("interval_years must be > 0 for log offset.")

    bin_col = f"{X_col}_bin"

    model = smf.gee(
        f"{Y_col} ~ {X_col}",
        groups="id",
        data=df2,
        family=sm.families.Poisson(),
        cov_struct=sm.cov_struct.Exchangeable(),
    )
    res = model.fit()

    beta = res.params[X_col]
    ci = res.conf_int().loc[X_col]
    p = res.pvalues[X_col]

    irr = np.exp(beta)
    ci_low = np.exp(ci[0])
    ci_high = np.exp(ci[1])
    print(f"{Y_col} ~ {X_col}")
    return p, irr, ci_low, ci_high

def format_p(p):
    if p < 0.00000001:
        return "p<0.00000001"
    elif p < 0.0000001:
        return "p<0.0000001"
    elif p < 0.000001:
        return "p<0.000001"
    elif p < 0.00001:
        return "p<0.00001"
    elif p < 0.0001:
        return "p<0.0001"
    elif p < 0.001:
        return "p<0.001"
    elif p < 0.01:
        return "p<0.01"
    elif p < 0.05:
        return "p<0.05"
    else:
        return f"p={p:.3f}"

In [1145]:
df_gee = pd.DataFrame(columns=['model','hospitalization','OR', 'OR_low', 'OR_high', 'p-val']) 

In [1146]:
######## any hospitalization

In [1147]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_hbf', 'hos_days')
df_gee.loc[len(df_gee)] = ['raw_hbf', 'Any', irr,ci_low, ci_high, format_p(p)]
print("\nGEE raw hbf result: Any hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

hos_days ~ low_hbf

GEE raw hbf result: Any hospitalization
OR = 2.18 (1.51-3.15), p = 3.491e-05


In [1148]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_adj_hbf', 'hos_days')
df_gee.loc[len(df_gee)] = ['adj_hbf', 'Any', irr,ci_low, ci_high, format_p(p)]
print("\nGEE adjusted hbf result: Any hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

hos_days ~ low_adj_hbf

GEE adjusted hbf result: Any hospitalization
OR = 2.42 (1.69-3.46), p = 1.333e-06


In [1149]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_sim_hbf', 'hos_days')
df_gee.loc[len(df_gee)] = ['predict_adj_hbf', 'Any', irr,ci_low, ci_high, format_p(p)]
print("\nGEE predicted adj hbf result: Any hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

hos_days ~ low_sim_hbf

GEE predicted adj hbf result: Any hospitalization
OR = 1.98 (1.32-2.98), p = 0.0009371


In [1150]:
############# VOC hospitalization

In [1151]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_hbf', 'voc')
df_gee.loc[len(df_gee)] = ['raw_hbf', 'VOC', irr,ci_low, ci_high, format_p(p)]
print("\nGEE raw hbf result: VOC hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

voc ~ low_hbf

GEE raw hbf result: VOC hospitalization
OR = 4.91 (2.83-8.51), p = 1.383e-08


In [1152]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_adj_hbf', 'voc')
df_gee.loc[len(df_gee)] = ['adj_hbf', 'VOC', irr,ci_low, ci_high, format_p(p)]
print("\nGEE adj hbf result: VOC hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

voc ~ low_adj_hbf

GEE adj hbf result: VOC hospitalization
OR = 4.87 (2.84-8.34), p = 8.2e-09


In [1153]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_sim_hbf', 'voc')
df_gee.loc[len(df_gee)] = ['predict_adj_hbf', 'VOC', irr,ci_low, ci_high, format_p(p)]
print("\nGEE predicted adj hbf result: VOC hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

voc ~ low_sim_hbf

GEE predicted adj hbf result: VOC hospitalization
OR = 4.52 (2.40-8.51), p = 2.961e-06


In [1154]:
############# ACS hospitalization

In [1155]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_hbf', 'acs')
df_gee.loc[len(df_gee)] = ['raw_hbf', 'ACS', irr,ci_low, ci_high, format_p(p)]
print("\nGEE raw hbf result: ACS hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

acs ~ low_hbf

GEE raw hbf result: ACS hospitalization
OR = 1.45 (0.56-3.78), p = 0.4467


In [1156]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_adj_hbf', 'acs')
df_gee.loc[len(df_gee)] = ['adj_hbf', 'ACS', irr,ci_low, ci_high, format_p(p)]
print("\nGEE adj hbf result: ACS hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

acs ~ low_adj_hbf

GEE adj hbf result: ACS hospitalization
OR = 1.14 (0.53-2.47), p = 0.7309


In [1157]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_sim_hbf', 'acs')
df_gee.loc[len(df_gee)] = ['predict_adj_hbf', 'ACS', irr,ci_low, ci_high, format_p(p)]
print("\nGEE predicted adj hbf result: ACS hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

acs ~ low_sim_hbf

GEE predicted adj hbf result: ACS hospitalization
OR = 1.50 (0.65-3.46), p = 0.3476


In [1158]:
########## VOC + ACS

In [1159]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_hbf', 'voc_acs')
df_gee.loc[len(df_gee)] = ['raw_hbf', 'VOC+ACS', irr,ci_low, ci_high, format_p(p)]
print("\nGEE raw hbf result: VOC/ACS hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

voc_acs ~ low_hbf

GEE raw hbf result: VOC/ACS hospitalization
OR = 3.00 (1.69-5.33), p = 0.0001873


In [1160]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_adj_hbf', 'voc_acs')
df_gee.loc[len(df_gee)] = ['adj_hbf', 'VOC+ACS', irr,ci_low, ci_high, format_p(p)]
print("\nGEE adj hbf result: VOC/ACS hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

voc_acs ~ low_adj_hbf

GEE adj hbf result: VOC/ACS hospitalization
OR = 2.83 (1.73-4.65), p = 3.818e-05


In [1161]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_sim_hbf', 'voc_acs')
df_gee.loc[len(df_gee)] = ['predict_adj_hbf', 'VOC+ACS', irr,ci_low, ci_high, format_p(p)]
print("\nGEE predicted adj hbf result: VOC/ACS hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

voc_acs ~ low_sim_hbf

GEE predicted adj hbf result: VOC/ACS hospitalization
OR = 2.94 (1.69-5.13), p = 0.000144


In [1162]:
############# FEVER hospitalization

In [1163]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_hbf', 'fever')
df_gee.loc[len(df_gee)] = ['raw_hbf', 'FEVER', irr,ci_low, ci_high, format_p(p)]
print("\nGEE raw hbf result: FEVER hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

fever ~ low_hbf

GEE raw hbf result: FEVER hospitalization
OR = 1.77 (0.91-3.43), p = 0.09122


In [1164]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_adj_hbf', 'fever')
df_gee.loc[len(df_gee)] = ['adj_hbf', 'FEVER', irr,ci_low, ci_high, format_p(p)]
print("\nGEE adj hbf result: FEVER hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

fever ~ low_adj_hbf

GEE adj hbf result: FEVER hospitalization
OR = 3.40 (1.81-6.38), p = 0.0001424


In [1165]:
p, irr, ci_low, ci_high =  gee_pro(df, 'low_sim_hbf', 'fever')
df_gee.loc[len(df_gee)] = ['predict_adj_hbf', 'FEVER', irr,ci_low, ci_high, format_p(p)]
print("\nGEE predicted adj hbf result: FEVER hospitalization")
print(f"OR = {irr:.2f} ({ci_low:.2f}-{ci_high:.2f}), p = {p:.4g}")

fever ~ low_sim_hbf

GEE predicted adj hbf result: FEVER hospitalization
OR = 1.80 (0.84-3.84), p = 0.1279


In [1166]:
df_gee

,model,hospitalization,OR,OR_low,OR_high,p-val
0,raw_hbf,Any,2.180872,1.507544,3.154933,p<0.0001
1,adj_hbf,Any,2.419185,1.690956,3.461035,p<0.00001
2,predict_adj_hbf,Any,1.984986,1.322451,2.979444,p<0.001
3,raw_hbf,VOC,4.912157,2.834970,8.511305,p<0.0000001
4,adj_hbf,VOC,4.866708,2.841602,8.335032,p<0.00000001
5,predict_adj_hbf,VOC,4.521335,2.401390,8.512765,p<0.00001
6,raw_hbf,ACS,1.450593,0.556490,3.781235,p=0.447
7,adj_hbf,ACS,1.144451,0.530517,2.468854,p=0.731
8,predict_adj_hbf,ACS,1.495391,0.645775,3.462809,p=0.348
9,raw_hbf,VOC+ACS,2.997952,1.685178,5.333391,p<0.001
